# 03: Social Signal Processing + ML Models

**Objective**: Extract trend signals from unstructured data using three
pre-trained / fine-tuned transformer models plus Google Trends.

| Pipeline | Model | Task | Training? |
|----------|-------|------|-----------|
| A | `cardiffnlp/twitter-roberta-base-sentiment-latest` | Fashion sentiment (LoRA fine-tuned) | Yes — LoRA |
| B | `facebook/bart-large-mnli` | Zero-shot style tribe classification | No — inference only |
| C | `sentence-transformers/all-MiniLM-L6-v2` | Semantic embeddings → HDBSCAN clusters | No — inference only |
| D | Google Trends (pytrends) | Search interest momentum | N/A |

**Output**: `social_signal_scorecard.parquet`, one row per category with
sentiment, style buzz, cluster strength, and trend momentum signals.

---

## 3.1 Setup

In [ ]:
import sys, os, logging, warnings
warnings.filterwarnings("ignore")

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = "/content/drive/MyDrive/FashionTrendAnalyzer"
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.system("pip install -q peft pytrends hdbscan umap-learn pydantic sentence-transformers")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import torch

from src.config import (
    DATA_PROCESSED, MODELS_DIR, RANDOM_SEED,
    reviews_config, sentiment_model_config, zero_shot_config, embedding_config,
    google_trends_config,
)
from src.data_loader import load_processed, save_processed
from src.social_signals import (
    map_reviews_to_category,
    aggregate_sentiment_by_category,
    aggregate_style_distribution,
    aggregate_cluster_strength,
    build_social_signal_scorecard,
)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
reviews = load_processed("reviews_clean")
reviews = map_reviews_to_category(reviews)

print(f"Reviews: {reviews.shape}")
print(f"Categories: {reviews['category'].nunique()}")
print(reviews["category"].value_counts())

---
## 3.2 Pipeline A: Fine-Tuned Fashion Sentiment (LoRA)

We fine-tune `cardiffnlp/twitter-roberta-base-sentiment-latest` on the fashion
reviews using LoRA (rank=16, alpha=32). This adapts the generic sentiment model
to understand fashion-domain language like "flattering", "runs small",
"obsessed with this print".

**Labels**: 0=negative (rating≤2), 1=neutral (rating=3), 2=positive (rating≥4 & recommended)

In [ ]:
from sklearn.model_selection import train_test_split
from src.models import (
    prepare_sentiment_labels,
    fine_tune_sentiment_model,
    load_fine_tuned_sentiment,
    predict_sentiment,
    baseline_vader_sentiment,
)

# Prepare labels
reviews["sentiment_label"] = prepare_sentiment_labels(
    reviews,
    rating_col=reviews_config.rating_column,
    recommend_col=reviews_config.recommend_column,
)

print("Label distribution:")
print(reviews["sentiment_label"].value_counts().sort_index())
print(f"  0=negative, 1=neutral, 2=positive")

In [ ]:
# Train/validation split
texts = reviews[reviews_config.text_column].tolist()
labels = reviews["sentiment_label"].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels,
    test_size=sentiment_model_config.test_size,
    stratify=labels,
    random_state=RANDOM_SEED,
)
print(f"Train: {len(train_texts):,} | Val: {len(val_texts):,}")

In [ ]:
# Fine-tune
adapter_path = MODELS_DIR / "adapter_config.json"

if adapter_path.exists():
    print("Fine-tuned adapter found — loading from disk.")
    model_ft, tokenizer_ft = load_fine_tuned_sentiment()
else:
    print("No adapter found — starting fine-tuning...")
    model_ft, tokenizer_ft = fine_tune_sentiment_model(
        train_texts, train_labels,
        val_texts, val_labels,
    )
    print("Fine-tuning complete.")

In [ ]:
# Evaluate: fine-tuned vs baseline VADER
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Fine-tuned predictions on validation set
ft_probs = predict_sentiment(val_texts, model_ft, tokenizer_ft)
ft_preds = np.argmax(ft_probs, axis=-1)

# VADER baseline
vader_scores = baseline_vader_sentiment(val_texts)
vader_preds = np.where(vader_scores < -0.05, 0, np.where(vader_scores > 0.05, 2, 1))

val_labels_arr = np.array(val_labels)

metrics = {
    "Fine-tuned RoBERTa (LoRA)": {
        "Accuracy": accuracy_score(val_labels_arr, ft_preds),
        "F1 (macro)": f1_score(val_labels_arr, ft_preds, average="macro"),
    },
    "VADER Baseline": {
        "Accuracy": accuracy_score(val_labels_arr, vader_preds),
        "F1 (macro)": f1_score(val_labels_arr, vader_preds, average="macro"),
    },
}

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
for name, m in metrics.items():
    print(f"\n{name}:")
    for k, v in m.items():
        print(f"  {k}: {v:.4f}")

print("\n--- Fine-tuned RoBERTa Classification Report ---")
print(classification_report(val_labels_arr, ft_preds, target_names=["Negative", "Neutral", "Positive"]))

In [ ]:
from src.viz import sentiment_comparison_bar

fig = sentiment_comparison_bar(metrics)
fig.show()

In [ ]:
# Run sentiment on ALL reviews
all_probs = predict_sentiment(texts, model_ft, tokenizer_ft)
# Sentiment score: weighted probability toward positive (0=negative, 1=positive scale)
reviews["sentiment_score"] = all_probs[:, 2] - all_probs[:, 0]  # range [-1, 1]
reviews["sentiment_pred"] = np.argmax(all_probs, axis=-1)

print(f"Sentiment score stats:")
print(reviews["sentiment_score"].describe())

---
## 3.3 Pipeline B: Zero-Shot Style Tribe Classification

Using `facebook/bart-large-mnli` to classify each review into a style tribe
**without any labeled training data**. This creates a dimension that does NOT
exist in the raw data, the key insight for trend intelligence.

Style tribes: minimalist, streetwear, bohemian, preppy, athleisure, quiet luxury,
Y2K revival, oversized comfort, bold prints, sustainable basics

In [ ]:
from src.models import classify_styles_zero_shot

# Run zero-shot on all reviews (batched for efficiency)
print(f"Classifying {len(texts):,} reviews into {len(zero_shot_config.style_labels)} style tribes...")
print(f"Labels: {zero_shot_config.style_labels}")

style_results = classify_styles_zero_shot(texts, batch_size=16)
print(f"\nDone. Shape: {style_results.shape}")
style_results.head(5)

In [ ]:
# Style tribe distribution
style_dist = style_results["style_label"].value_counts()

fig = px.bar(
    style_dist.reset_index(),
    x="count", y="style_label",
    orientation="h",
    title="Zero-Shot Style Tribe Distribution Across All Reviews",
    labels={"count": "Number of Reviews", "style_label": "Style Tribe"},
    template="plotly_white",
    color="count",
    color_continuous_scale="Sunset",
)
fig.update_layout(height=400, width=700, yaxis=dict(autorange="reversed"))
fig.show()

In [ ]:
# Confidence distribution
fig = px.histogram(
    style_results, x="confidence",
    nbins=50,
    title="Zero-Shot Classification Confidence Distribution",
    labels={"confidence": "Confidence Score"},
    template="plotly_white",
)
fig.add_vline(x=zero_shot_config.confidence_threshold, line_dash="dash",
              annotation_text="Threshold")
fig.update_layout(height=350, width=700)
fig.show()

print(f"Reviews above confidence threshold ({zero_shot_config.confidence_threshold}): "
      f"{(style_results['confidence'] >= zero_shot_config.confidence_threshold).mean():.1%}")

In [ ]:
# Style tribe × Category heatmap
reviews["style_label"] = style_results["style_label"].values
reviews["style_confidence"] = style_results["confidence"].values

cross = pd.crosstab(reviews["category"], reviews["style_label"], normalize="index")

fig = px.imshow(
    cross,
    color_continuous_scale="YlOrRd",
    title="Style Tribe Distribution by Product Category",
    labels=dict(color="Proportion"),
    template="plotly_white",
    aspect="auto",
)
fig.update_layout(height=400, width=900)
fig.show()

---
## 3.4 Pipeline C: Semantic Trend Clustering

Embed all reviews with MiniLM, reduce dimensions with UMAP, then cluster
with HDBSCAN to discover emergent trend themes the predefined labels might miss.

In [ ]:
from src.models import compute_embeddings, cluster_embeddings, label_clusters
from src.viz import umap_cluster_scatter

print(f"Computing embeddings for {len(texts):,} reviews...")
embeddings = compute_embeddings(texts)
print(f"Embedding shape: {embeddings.shape}")

In [ ]:
umap_2d, cluster_labels = cluster_embeddings(embeddings)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
noise_pct = (cluster_labels == -1).mean()

print(f"Clusters found: {n_clusters}")
print(f"Noise points: {noise_pct:.1%}")

In [ ]:
# Auto-label clusters via TF-IDF
cluster_label_map = label_clusters(texts, cluster_labels)

print("\nDiscovered trend clusters:")
for cid, label in sorted(cluster_label_map.items()):
    count = (cluster_labels == cid).sum()
    if cid != -1:
        print(f"  Cluster {cid:2d} ({count:5,} reviews): {label}")

In [ ]:
# Interactive UMAP visualization
fig = umap_cluster_scatter(umap_2d, cluster_labels, cluster_label_map, texts)
fig.show()

---
## 3.5 Pipeline D: Google Trends Momentum

Pull search interest data for fashion keywords and compute trend momentum
(linear slope over trailing 12 weeks) with breakout detection.

In [ ]:
from src.google_trends import fetch_trends_data, compute_trend_momentum, aggregate_trends_by_category
from src.viz import trend_momentum_timeseries

# Fetch Google Trends data
# Note: pytrends may hit rate limits; this cell is retry-safe
try:
    trends_raw = fetch_trends_data()
    print(f"Trends data: {trends_raw.shape}")
    trends_raw.tail(3)
except Exception as e:
    print(f"Google Trends fetch failed (rate limit or network issue): {e}")
    print("Using empty trends data — downstream pipeline handles this gracefully.")
    trends_raw = pd.DataFrame()

In [ ]:
# Compute momentum and breakouts
momentum_df = compute_trend_momentum(trends_raw)

if not momentum_df.empty:
    print(f"\nTop 10 by momentum:")
    print(momentum_df.nlargest(10, "momentum")[["keyword", "momentum", "current_interest", "breakout_flag"]].to_string(index=False))

    breakouts = momentum_df[momentum_df["breakout_flag"]]
    if len(breakouts) > 0:
        print(f"\nBreakout keywords ({len(breakouts)}):")
        print(breakouts[["keyword", "current_interest", "momentum"]].to_string(index=False))
else:
    print("No momentum data available.")

In [ ]:
# Time series visualization
fig = trend_momentum_timeseries(trends_raw)
fig.show()

In [ ]:
# Aggregate trends to category level
trends_by_category = aggregate_trends_by_category(momentum_df)
print(f"Trends by category: {len(trends_by_category)} categories")
trends_by_category

---
## 3.6 Build Social Signal Scorecard

Aggregate all four pipelines into a single scorecard at the category level.

In [ ]:
# Aggregate each pipeline
sentiment_agg = aggregate_sentiment_by_category(reviews, sentiment_col="sentiment_score")
style_agg = aggregate_style_distribution(style_results, reviews)
cluster_agg = aggregate_cluster_strength(cluster_labels, reviews["category"], cluster_label_map)

print("Sentiment agg:", sentiment_agg.shape)
print("Style agg:", style_agg.shape)
print("Cluster agg:", cluster_agg.shape)
print("Trends agg:", trends_by_category.shape if not trends_by_category.empty else "empty")

In [ ]:
# Merge into unified scorecard
social_scorecard = build_social_signal_scorecard(
    sentiment_agg=sentiment_agg,
    style_agg=style_agg,
    cluster_agg=cluster_agg,
    trends_agg=trends_by_category if not trends_by_category.empty else None,
)

print(f"\nSocial Signal Scorecard: {social_scorecard.shape}")
social_scorecard

In [ ]:
# Save scorecard
save_processed(social_scorecard, "social_signal_scorecard")
print(f"Saved social_signal_scorecard.parquet — {len(social_scorecard)} categories")